# 07 · pgvector walkthrough

This is an ungraded deep dive into **pgvector**, the PostgreSQL extension behind this lab's *vector store B* and the *governed-metric source of record*. It's also the engine under Bedrock Knowledge Bases on Aurora and many agent memory stores. Agents care about it because one database can hold:
- vectors, next to their relational truth;
- filters and permissions;
- transactions.

| Section | What you'll see |
|---|---|
| 1. Types and distances | `vector`, `halfvec`, `sparsevec`; the operators `<->`, `<=>`, `<#>`; what normalised vectors change |
| 2. Exact vs approximate search | `EXPLAIN`, HNSW, and why the operator class must match the query |
| 3. Recall versus speed | HNSW `ef_search` and IVFFlat `probes`, measured against exact search on 20,000 vectors |
| 4. Filtered search | Why `WHERE tenant = …` can return *too few* rows, and pgvector 0.8 iterative scans |
| 5. Hybrid search in one SQL query | Full-text (`tsvector`) + vectors, fused with RRF (the same idea as goal 4) |
| 6. Agentic patterns | Agent memory with metadata, and row-level security per tenant |

Everything happens in a scratch schema `walkthrough`, which the last cell drops. The lab's own tables (`coa_chunks`, `quarterly_capital`, `cargo_experience`) are never touched. It uses the same connection as the lab (Aurora, or your local PostgreSQL, from `.env`) and 40 insurance clauses embedded once with Titan and cached in `notebooks/data/`.

In [ ]:
import os, sys, time, random
sys.path.insert(0, os.path.abspath('.'))
import labkit
from src.pgvector_client import PgVectorStore

pg = PgVectorStore()          # the lab's connection settings from .env; autocommit is on
conn = pg.conn

def sql(q, params=None, show=True):
    with conn.cursor() as cur:
        cur.execute(q, params)
        rows = cur.fetchall() if cur.description else []
    if show:
        for r in rows:
            print(r)
    return rows

def explain(q, params=None):
    for (line,) in sql('EXPLAIN (ANALYZE, COSTS OFF, TIMING OFF) ' + q, params, show=False):
        print('   ', line)

def vec(v):
    """Python list -> pgvector text literal '[x1,x2,...]'."""
    return '[' + ','.join(f'{x:.7f}' for x in v) + ']'

sql('CREATE EXTENSION IF NOT EXISTS vector', show=False)
PGV = tuple(int(x) for x in sql("SELECT extversion FROM pg_extension WHERE extname = 'vector'", show=False)[0][0].split('.'))
print('PostgreSQL:', sql('SHOW server_version', show=False)[0][0], '| pgvector:', '.'.join(map(str, PGV)))
sql("SELECT version FROM pg_available_extension_versions WHERE name = 'vector' ORDER BY version", show=False)
sql('DROP SCHEMA IF EXISTS walkthrough CASCADE; CREATE SCHEMA walkthrough', show=False)
print('scratch schema walkthrough ready')

## 1 · Types and distances

A `vector(n)` column stores `n` single-precision floats, and the dimension is enforced. pgvector 0.7 added two more types:
- `halfvec`: 16-bit floats, half the storage, and indexable up to 4,000 dimensions;
- `sparsevec`: only the non-zero entries, for sparse or lexical embeddings.

For comparison, plain `vector` can be indexed up to 2,000 dimensions.

In [ ]:
sql("SELECT '[1,2,3]'::vector AS v, vector_dims('[1,2,3]'::vector) AS dims, '[3,4]'::vector <-> '[0,0]'::vector AS l2_distance")
try:
    sql("SELECT '[1,2,3]'::vector(2)")
except Exception as e:
    print('dimension check:', str(e).splitlines()[0])
if PGV >= (0, 7, 0):
    sql("SELECT '[1.5,2.25]'::halfvec AS half, '{1:0.5,7:1}/10'::sparsevec AS sparse")

Load the corpus: 40 clauses, each with a tenant (broker), a line of business, a Titan embedding and a generated full-text column.

In [ ]:
clauses = labkit.corpus()
embs = labkit.embed([c['text'] for c in clauses])
DIMS = len(embs[0])
sql(f'''CREATE TABLE walkthrough.clauses (
          clause_id text PRIMARY KEY,
          tenant    text NOT NULL,
          lob       text NOT NULL,
          body      text NOT NULL,
          embedding vector({DIMS}) NOT NULL,
          tsv       tsvector GENERATED ALWAYS AS (to_tsvector('english', body)) STORED)''')
with conn.cursor() as cur:
    cur.executemany('INSERT INTO walkthrough.clauses (clause_id, tenant, lob, body, embedding) VALUES (%s, %s, %s, %s, %s::vector)',
                    [(c['clause_id'], c['tenant'], c['lob'], c['text'], vec(e)) for c, e in zip(clauses, embs)])
sql('SELECT lob, count(*) FROM walkthrough.clauses GROUP BY lob ORDER BY lob')

The three distance operators:

| Operator | Distance | Index operator class | Note |
|---|---|---|---|
| `<->` | Euclidean (L2) | `vector_l2_ops` | |
| `<=>` | cosine distance = 1 − cosine similarity | `vector_cosine_ops` | |
| `<#>` | **negative** inner product | `vector_ip_ops` | Negative so that `ORDER BY … ASC` still means "best first" |

Titan V2 returns **normalised** vectors (`normalize: true`). For unit vectors, L2² = 2 − 2·cos and inner product = cosine, so all three operators produce **the same ranking**. Their values differ, though, and the index you build must match the operator you query with.

In [ ]:
Q = 'Is flood damage covered?'
qv = vec(labkit.embed([Q])[0])
sql('''SELECT clause_id,
              round((embedding <-> %(q)s::vector)::numeric, 4)       AS l2,
              round((embedding <=> %(q)s::vector)::numeric, 4)       AS cosine_distance,
              round((1 - (embedding <=> %(q)s::vector))::numeric, 4) AS cosine_similarity,
              round((embedding <#> %(q)s::vector)::numeric, 4)       AS neg_inner_product,
              left(body, 55) AS body
       FROM walkthrough.clauses
       ORDER BY embedding <=> %(q)s::vector
       LIMIT 5''', {'q': qv})

## 2 · Exact vs approximate search

**Without an index**, `ORDER BY embedding <=> q LIMIT k` is **exact**. PostgreSQL computes the distance to every row and sorts, so recall is 100% and the cost is linear in the number of rows.

An **HNSW** index is a navigable layered graph. It finds the approximate nearest neighbours in roughly logarithmic time.

The planner uses a vector index only when **the query's operator matches the index's operator class**, and the query has `ORDER BY <distance> LIMIT k`.

With only 40 rows a sequential scan is genuinely cheaper, so we switch scans off to *see* the index plan.

In [ ]:
Q_COS = 'SELECT clause_id FROM walkthrough.clauses ORDER BY embedding <=> %(q)s::vector LIMIT 5'
Q_L2  = 'SELECT clause_id FROM walkthrough.clauses ORDER BY embedding <-> %(q)s::vector LIMIT 5'
print('no index (exact):'); explain(Q_COS, {'q': qv})

sql('CREATE INDEX clauses_hnsw ON walkthrough.clauses USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64)')
sql('SET enable_seqscan = off')
print('\ncosine query, cosine index:'); explain(Q_COS, {'q': qv})
print('\nL2 query, cosine index (operator mismatch: the index cannot be used):'); explain(Q_L2, {'q': qv})
sql('RESET enable_seqscan')

The HNSW build parameters:
- `m`: links per node. Higher means better recall, more memory and slower builds.
- `ef_construction`: the candidate list size while building.

At query time, `hnsw.ef_search` (default 40) is the candidate list size, and the main recall/latency dial.

## 3 · Recall versus speed

Approximate means you trade **recall**, the fraction of the true top-k you get back, for speed. Measure it, don't assume it. The test uses 20,000 random 128-dimension vectors generated inside PostgreSQL, 20 random queries, and recall@10 against exact search.

In [ ]:
t0 = time.perf_counter()
sql('''CREATE TABLE walkthrough.synth AS
       SELECT i AS id,
              i % 50 AS tenant,
              (SELECT array_agg(random()::real) FROM generate_series(1, 128) WHERE i > 0)::vector(128) AS v
       FROM generate_series(1, 20000) AS i''')
print(f'20,000 vectors generated in {time.perf_counter() - t0:.1f}s')

rng = random.Random(7)
QS = [vec([rng.random() for _ in range(128)]) for _ in range(20)]

def top_ids(q, k=10, settings=()):
    with conn.cursor() as cur:
        for s in settings:
            cur.execute(s)
        cur.execute('SELECT id FROM walkthrough.synth ORDER BY v <=> %s::vector LIMIT %s', (q, k))
        ids = [r[0] for r in cur.fetchall()]
        cur.execute('RESET ALL')
    return ids

t0 = time.perf_counter()
TRUTH = [top_ids(q) for q in QS]                     # no index yet: exact
EXACT_MS = (time.perf_counter() - t0) * 1000 / len(QS)
print(f'exact search: {EXACT_MS:.1f} ms/query (recall 1.00 by definition)')

def measure(label, settings):
    t0 = time.perf_counter()
    got = [top_ids(q, settings=settings) for q in QS]
    ms = (time.perf_counter() - t0) * 1000 / len(QS)
    recall = sum(len(set(a) & set(b)) for a, b in zip(got, TRUTH)) / (10 * len(QS))
    print(f'{label:28} recall@10 = {recall:.2f}   {ms:6.1f} ms/query')

In [ ]:
t0 = time.perf_counter()
sql('CREATE INDEX synth_hnsw ON walkthrough.synth USING hnsw (v vector_cosine_ops) WITH (m = 16, ef_construction = 64)')
print(f'HNSW build: {time.perf_counter() - t0:.1f}s')
for ef in (10, 40, 100, 200):
    measure(f'HNSW ef_search={ef}', ('SET enable_seqscan = off', f'SET hnsw.ef_search = {ef}'))

sql('DROP INDEX walkthrough.synth_hnsw')
t0 = time.perf_counter()
sql('CREATE INDEX synth_ivf ON walkthrough.synth USING ivfflat (v vector_cosine_ops) WITH (lists = 100)')
print(f'\nIVFFlat build (lists=100): {time.perf_counter() - t0:.1f}s')
for probes in (1, 5, 10, 30):
    measure(f'IVFFlat probes={probes}', ('SET enable_seqscan = off', f'SET ivfflat.probes = {probes}'))

What to take away:
- **HNSW**: better recall at a given speed, works on an empty table, and slower and bigger to build. It's the usual default.
- **IVFFlat**: clusters the data into `lists` buckets and searches `probes` of them. It's fast to build, but build it **after** loading data (the clusters come from the data). Rules of thumb: `lists ≈ rows / 1000`, `probes ≈ √lists`.
- **Uniform random vectors are a worst case.** Real embeddings cluster, so recall is usually higher. Always measure on *your* data with *your* queries.

## 4 · Filtered search: the agent's most common query

Agents rarely search "everything". They search *this tenant's* documents, or *this customer's* memories. With an ANN index, PostgreSQL:
1. asks the index for the `ef_search` nearest candidates;
2. **then** applies the `WHERE` clause.

With a selective filter, most candidates are thrown away and you get **fewer than k rows back**. Each tenant here holds only 2% of the rows.

In [ ]:
sql('DROP INDEX walkthrough.synth_ivf')
sql('CREATE INDEX synth_hnsw ON walkthrough.synth USING hnsw (v vector_cosine_ops) WITH (m = 16, ef_construction = 64)')
q = QS[0]

def filtered(settings):
    with conn.cursor() as cur:
        for s in settings:
            cur.execute(s)
        cur.execute('SELECT id FROM walkthrough.synth WHERE tenant = 7 ORDER BY v <=> %s::vector LIMIT 10', (q,))
        n = len(cur.fetchall())
        cur.execute('RESET ALL')
    return n

print('rows for tenant 7:', sql('SELECT count(*) FROM walkthrough.synth WHERE tenant = 7', show=False)[0][0])
print('HNSW + WHERE tenant = 7, ef_search=40   ->', filtered(('SET enable_seqscan = off', 'SET hnsw.ef_search = 40')), 'rows (asked for 10)')
if PGV >= (0, 8, 0):
    print('same, with hnsw.iterative_scan        ->',
          filtered(('SET enable_seqscan = off', 'SET hnsw.ef_search = 40', "SET hnsw.iterative_scan = 'relaxed_order'")), 'rows')
else:
    print('iterative scans need pgvector 0.8.0 (Aurora PostgreSQL 16.6+ / 15.10+ ship it)')

Fixes for filtered vector search, from least to most effort:
- **Iterative index scans** (pgvector ≥ 0.8): `hnsw.iterative_scan = relaxed_order` (or `strict_order`) keeps walking the graph until `LIMIT` rows pass the filter.
- **Exact search on a small slice:** a B-tree index on the filter column plus no ANN at all. For a few thousand rows per tenant, exact search is fast *and* 100% recall. Agent memory per user is usually this case.
- **Partial indexes or partitioning** per tenant or category, so each ANN index only contains rows that match.
- Raise `ef_search`. That's a blunt tool: it costs latency on every query.

## 5 · Hybrid search in one SQL query

Goal 4 fused OpenSearch's BM25 and kNN lists in Python. PostgreSQL can do the same **in one statement**: full-text search (`tsvector` with `ts_rank_cd`) for the keyword list, `<=>` for the semantic list, and RRF in the `SELECT`.

In [ ]:
# plainto_tsquery ANDs every word ('piraci' & 'exclus' & 'sea'), which is too strict for retrieval:
# turn it into an OR query, as BM25 'match' does by default.
HYBRID = '''
WITH q AS (
    SELECT to_tsquery('english', replace(plainto_tsquery('english', %(qt)s)::text, ' & ', ' | ')) AS query
), semantic AS (
    SELECT clause_id, row_number() OVER (ORDER BY embedding <=> %(qv)s::vector) AS rank
    FROM walkthrough.clauses
    ORDER BY embedding <=> %(qv)s::vector
    LIMIT 20
), keyword AS (
    SELECT clause_id, row_number() OVER (ORDER BY ts_rank_cd(tsv, q.query) DESC) AS rank
    FROM walkthrough.clauses, q
    WHERE tsv @@ q.query
    ORDER BY ts_rank_cd(tsv, q.query) DESC
    LIMIT 20
)
SELECT clause_id, s.rank AS semantic_rank, k.rank AS keyword_rank,
       round((coalesce(1.0 / (60 + s.rank), 0) + coalesce(1.0 / (60 + k.rank), 0))::numeric, 5) AS rrf,
       left(c.body, 60) AS body
FROM semantic s
FULL OUTER JOIN keyword k USING (clause_id)
JOIN walkthrough.clauses c USING (clause_id)
ORDER BY rrf DESC
LIMIT 5'''

for question in ('piracy exclusions at sea', 'what happens if hackers lock our files'):
    print(f'\n=== {question}')
    print('lexemes:', sql("SELECT plainto_tsquery('english', %s)::text", (question,), show=False)[0][0])
    sql(HYBRID, {'qv': vec(labkit.embed([question])[0]), 'qt': question})

Read the two result tables:
- **"piracy exclusions at sea"**: `piraci` matches MC-01 ("Cargo lost to **piracy**…") and `sea` matches MC-02. Semantic search also ranks MC-01 high, so MC-01 collects **two** RRF contributions and wins.
  - Notice that `exclus` (from *exclusions*) does **not** match *excluded* (`exclud`). Stemming is algorithmic, not semantic.
- **"what happens if hackers lock our files"**: the only keyword hit is probably AU-06 ("…when the van is **locked**"), a word match with the wrong meaning. The semantic list finds CY-01 (ransomware). Fusion keeps the keyword hit visible, but agreement is impossible here, so the semantic rank decides.

That's the case for hybrid retrieval in agents: exact identifiers and jargon on one side, paraphrased intent on the other, with neither trusted alone. Notebook 08 shows the same analyzer effects in OpenSearch.

## 6 · Agentic patterns

### 6a · Agent memory: vectors next to their metadata

An agent's long-term memory is a table: who it's about (`actor`), what kind of memory it is, the text, the embedding and a timestamp. Retrieval filters on the **trusted** actor id, which comes from the session or token, never from the model. It then ranks by meaning. Per-actor sets are small, so a B-tree on `actor` plus exact distance is exact *and* fast.

In [ ]:
sql(f'''CREATE TABLE walkthrough.agent_memory (
          memory_id  bigserial PRIMARY KEY,
          actor      text NOT NULL,
          kind       text NOT NULL,
          content    text NOT NULL,
          embedding  vector({DIMS}) NOT NULL,
          created_at timestamptz NOT NULL DEFAULT now());
        CREATE INDEX ON walkthrough.agent_memory (actor);''')
memories = [('alex', 'preference', 'Prefers email over phone calls.'),
            ('alex', 'fact', 'Owns a warehouse in Rotterdam insured through Harbor.'),
            ('alex', 'fact', 'Had a ransomware incident in 2025.'),
            ('sam', 'preference', 'Wants all quotes in euros.'),
            ('sam', 'fact', 'Runs a refrigerated shipping business.')]
with conn.cursor() as cur:
    for (actor, kind, content), e in zip(memories, labkit.embed([m[2] for m in memories])):
        cur.execute('INSERT INTO walkthrough.agent_memory (actor, kind, content, embedding) VALUES (%s, %s, %s, %s::vector)',
                    (actor, kind, content, vec(e)))

def recall_memories(actor, question, k=2):
    print(f'\n[{actor}] {question}')
    sql('''SELECT kind, content, round((1 - (embedding <=> %(q)s::vector))::numeric, 3) AS similarity
           FROM walkthrough.agent_memory WHERE actor = %(actor)s
           ORDER BY embedding <=> %(q)s::vector LIMIT %(k)s''',
        {'q': vec(labkit.embed([question])[0]), 'actor': actor, 'k': k})

recall_memories('alex', 'How should we contact this customer?')
recall_memories('alex', 'Any previous cyber claims?')
recall_memories('sam', 'What does the customer ship?')

Compare this with goal 1. DynamoDB memory is **ordered by time** (turns), and retrieval is by key. A pgvector memory is retrieved **by meaning**. Real agents often use both: short-term conversational turns in a key-value store, and long-term facts and preferences in a vector table.

### 6b · Row-level security: tenant isolation the agent can't bypass

If several brokers' agents share one table, don't trust the agent's SQL to add `WHERE tenant = …`. Let PostgreSQL enforce it:
- **row-level security** (RLS) with a policy on a session setting that your application sets from the authenticated caller;
- `FORCE` makes the policy apply to the table owner as well.

The policy fails closed: with no tenant set, no rows are visible.

*On a local PostgreSQL where you connect as a **superuser**, RLS is always bypassed. Aurora's master user is `rds_superuser`, not a true superuser, so the policy applies.*

In [ ]:
sql('''ALTER TABLE walkthrough.clauses ENABLE ROW LEVEL SECURITY;
       ALTER TABLE walkthrough.clauses FORCE ROW LEVEL SECURITY;
       CREATE POLICY tenant_isolation ON walkthrough.clauses
           USING (tenant = current_setting('app.tenant', true));''')
try:
    print('no tenant set:', sql('SELECT count(*) FROM walkthrough.clauses', show=False)[0][0], 'rows visible')
    for tenant in ('harbor', 'summit'):
        sql(f"SET app.tenant = '{tenant}'")
        print(f'\napp.tenant = {tenant}:')
        sql('''SELECT tenant, clause_id, left(body, 60) FROM walkthrough.clauses
               ORDER BY embedding <=> %(q)s::vector LIMIT 3''', {'q': vec(labkit.embed(['Is flood damage covered?'])[0])})
finally:
    sql('RESET app.tenant; ALTER TABLE walkthrough.clauses NO FORCE ROW LEVEL SECURITY; ALTER TABLE walkthrough.clauses DISABLE ROW LEVEL SECURITY')

Same question, different tenant, different answers. In an agent, set `app.tenant` from the verified identity (for example the JWT claims, as in the Securing Agent Identities kit) at the start of each request. Use a dedicated database role whose only privileges are on these tables.

### 6c · Governed SQL

The lab's `src/metric_service.py` is the third pattern. The graph stores a metric's *definition*, but execution uses only an **allow-listed, parameterised, read-only** query. A vector database next to your relational truth makes this natural: retrieval finds the *right metric*, and SQL computes the *right number*.

### Aurora notes

- **Versions.** pgvector ships with Aurora PostgreSQL. `SELECT * FROM pg_available_extension_versions WHERE name = 'vector'` lists what your engine offers, and `ALTER EXTENSION vector UPDATE` upgrades it after an engine upgrade. HNSW needs 0.5+, `halfvec`/`sparsevec` 0.7+, and iterative scans 0.8+.
- **Memory.** HNSW performs best when the index fits in memory. `maintenance_work_mem` speeds up index builds. Aurora **Optimized Reads** (instances with local NVMe) help when it doesn't fit.
- **Bedrock Knowledge Bases** can use an Aurora table like `coa_chunks` as their vector store. The Databases for Agentic AI kit (D2, D6, D8, D10) covers that, including tuning faults.

## Clean up

In [ ]:
sql('DROP SCHEMA IF EXISTS walkthrough CASCADE')
pg.close()
print('scratch schema dropped; the lab tables are untouched')